In [ ]:
!pip install open3d datasets plyfile huggingface_hub plotly wandb

DOWNLOADING

In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY')
os.environ['HF_TOKEN']=userdata.get('HUGGINGFACE')


from huggingface_hub import hf_hub_download, snapshot_download

# Downloading the dataset
hf_hub_download(
    repo_id="BGLab/AgriField3D",
    filename="datasets/FielGrwon_ZeaMays_RawPCD_10k.zip",
    repo_type="dataset",
    local_dir="/content/MaizeField3D"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


datasets/FielGrwon_ZeaMays_RawPCD_10k.zi(…):   0%|          | 0.00/218M [00:00<?, ?B/s]

'/content/MaizeField3D/datasets/FielGrwon_ZeaMays_RawPCD_10k.zip'

UNZIPPING

In [ ]:
import zipfile, os

zip_path = "/content/MaizeField3D/datasets/FielGrwon_ZeaMays_RawPCD_10k.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("./MaizeField3D/extracted")

print("Unzipped!")

Unzipped!


In [ ]:
import os

data_path = "/content/MaizeField3D/extracted"

ply_files = []

for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith(".ply"):
            ply_files.append(os.path.join(root, file))

print("Total PLY files:", len(ply_files))

Total PLY files: 1045


VISUALIZING USING PLOTLY

In [ ]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

def visualize_plant(ply_file):
    pcd = o3d.io.read_point_cloud(ply_file)
    points = np.asarray(pcd.points)

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points[:,0],
                y=points[:,1],
                z=points[:,2],
                mode="markers",
                marker=dict(size=2)
            )
        ]
    )

    fig.update_layout(
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z"
        ),
        height=700
    )

    fig.show()

visualize_plant(ply_files[2])

INSPECTING COLORS AND NORMALS

In [ ]:
import open3d as o3d
import numpy as np

import open3d as o3d

def inspect_ply_contents(ply_file):
    pcd = o3d.io.read_point_cloud(ply_file)

    print("Points:", len(pcd.points))
    print("Has colors:", pcd.has_colors())
    print("Has normals:", pcd.has_normals())

inspect_ply_contents(ply_files[0])

Points: 10000
Has colors: True
Has normals: False


VISUALIZING THE COLORS

In [ ]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

import open3d as o3d
import numpy as np
import plotly.graph_objects as go

def visualize_three_maize(ply_files, spacing=1.5):
    fig = go.Figure()

    for i, file in enumerate(ply_files[:3]):
        pcd = o3d.io.read_point_cloud(file)

        points = np.asarray(pcd.points)
        colors = np.asarray(pcd.colors)

        # normalize plant
        points = points - points.min(axis=0)
        points = points / points.max()

        # offset plants side by side
        points[:,0] += i * spacing

        fig.add_trace(go.Scatter3d(
            x=points[:,0],
            y=points[:,1],
            z=points[:,2],
            mode="markers",
            marker=dict(
                size=2,
                color=(colors * 255).astype(int)
            )
        ))

    fig.update_layout(height=700)
    fig.show()


visualize_three_maize(ply_files)

In [ ]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

def plant_height_distribution(ply_files):
    heights = []

    for file in ply_files:
        pcd = o3d.io.read_point_cloud(file)
        points = np.asarray(pcd.points)
        height = points[:,2].max() - points[:,2].min()
        heights.append(height)

    fig = go.Figure(data=[go.Histogram(x=heights, nbinsx=30)])
    fig.update_layout(
        xaxis_title="Plant Height",
        yaxis_title="Number of Plants"
    )
    fig.show()


plant_height_distribution(ply_files)

1. The x-axis represents plant height and y-axis shows the number of plants.

2. The plant height distribution follows a bell-shaped pattern, where most maize plants cluster around the average height, while very short or very tall plants occur less frequently

3. Most plants fall between ~1.6 and 2.1 height units.

4. The peak around ~1.9 indicates the most common plant height.

5. Very few plants are extremely short (<1.2) or very tall (>2.5).

CONVERSIONS

POINT CLOUD TO MESH using Poisson

In [ ]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def visualize_original_and_mesh(ply_file):

    pcd = o3d.io.read_point_cloud(ply_file)
    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors)

    center = points.mean(axis=0)
    points = (points - center) / np.abs(points - center).max()
    pcd.points = o3d.utility.Vector3dVector(points)

    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.03, max_nn=50)
    )
    pcd.orient_normals_consistent_tangent_plane(100)

    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=8)

    densities = np.asarray(densities)
    mesh.remove_vertices_by_mask(densities < np.quantile(densities, 0.1))

    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)

    fig = make_subplots(rows=1, cols=2, specs=[[{"type":"scene"},{"type":"scene"}]])

    fig.add_trace(
        go.Scatter3d(
            x=points[:,0], y=points[:,1], z=points[:,2],
            mode="markers",
            marker=dict(size=2, color=(colors*255).astype(int)),
            name="Original"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Mesh3d(
            x=vertices[:,0], y=vertices[:,1], z=vertices[:,2],
            i=triangles[:,0], j=triangles[:,1], k=triangles[:,2],
            opacity=1
        ),
        row=1, col=2
    )

    fig.show()

visualize_original_and_mesh(ply_files[0])

Point cloud to mesh using Ball pivoting

In [ ]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def visualize_original_and_bpa_mesh(ply_file):

    pcd = o3d.io.read_point_cloud(ply_file)
    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors)

    center = points.mean(axis=0)
    points = (points - center) / np.abs(points - center).max()
    pcd.points = o3d.utility.Vector3dVector(points)

    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.03, max_nn=50)
    )

    distances = pcd.compute_nearest_neighbor_distance()
    avg_dist = np.mean(distances)

    radii = [avg_dist, avg_dist * 2]

    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        pcd,
        o3d.utility.DoubleVector(radii)
    )

    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)

    fig = make_subplots(rows=1, cols=2, specs=[[{"type":"scene"},{"type":"scene"}]])

    fig.add_trace(
        go.Scatter3d(
            x=points[:,0],
            y=points[:,1],
            z=points[:,2],
            mode="markers",
            marker=dict(size=2, color=(colors*255).astype(int)),
            name="Original"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Mesh3d(
            x=vertices[:,0],
            y=vertices[:,1],
            z=vertices[:,2],
            i=triangles[:,0],
            j=triangles[:,1],
            k=triangles[:,2],
            opacity=1
        ),
        row=1, col=2
    )

    fig.show()


visualize_original_and_bpa_mesh(ply_files[0])

#Why Ball pivoting give better approximation


Ball Pivoting reconstructs the surface directly from local point connectivity, while Poisson fits a global smooth implicit surface.

Ball Pivoting: preserves sharp geometry and thin structures (like maize leaves).

Poisson: smooths surfaces and fills gaps, which can blur thin plant structures.